# 사용자 정의 도구 만들기 (@tool)

`@tool` 데코레이터로 만든 도구는 LLM에게 **이름(`name`), 설명(`description`), 인자 스키마(`args`)** 세 가지로 전달된다. LLM은 이 정보만 보고 도구를 언제, 어떤 인자로 부를지 정하기 때문에, 설명을 얼마나 잘 쓰느냐가 도구 호출 품질을 좌우한다. 같은 함수를 두 가지 방식으로 만들어 LLM에게 전달되는 정보가 어떻게 달라지는지 비교한다.

## 1. 기본 @tool

함수 이름이 도구 이름이 되고, docstring이 도구 설명이 되고, 타입 힌트(`a: int`)가 인자 스키마가 된다. 이 경우 인자 스키마에는 타입만 있고 **각 인자가 무엇인지 설명은 없다.**

In [1]:
from langchain.tools import tool 

# @tool: 함수를 LLM이 호출할 수 있는 도구로 만든다.
@tool 
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b 

# 도구 이름(함수 이름), 설명(docstring), 인자 스키마(타입 힌트)를 확인
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## 2. docstring에서 인자 설명까지 가져오기 (parse_docstring=True)

`parse_docstring=True`를 주면 docstring의 `Args:` 부분을 읽어서 인자마다 설명을 붙여 준다. 출력을 보면 `description`에는 `Args:` 부분이 빠진 첫 줄만 남고, `args`의 `a`, `b`에 `'description': 'The first number.'`처럼 설명이 들어갔다.

In [4]:
# parse_docstring=True: docstring의 Args: 부분을 읽어 인자별 설명으로 넣는다. (Google 스타일 docstring 형식이어야 함)
@tool(parse_docstring=True)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers.
    
    Args:
        a: The first number.
        b: The second number.
    """
    return a * b 

print(multiply.name) 
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'description': 'The first number.', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number.', 'title': 'B', 'type': 'integer'}}


## 3. LLM에게 실제로 전달되는 JSON 스키마

`args_schema.model_json_schema()`로 보면 도구가 최종적으로 어떤 JSON 스키마가 되는지 알 수 있다. 도구 설명(`description`), 인자별 타입과 설명(`properties`), 필수 인자(`required`)가 들어 있고, 이 내용이 LLM 요청에 함께 실려 간다. (6.3.2의 코드 실행 도구에서는 같은 일을 `pydantic.Field(description=...)`로 했다.)

In [5]:
# 도구 인자 스키마를 JSON 스키마로 출력 (LLM에게 전달되는 형태)
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers.', 'properties': {'a': {'description': 'The first number.', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number.', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## 정리

| 방법 | 인자 설명 | 코드 |
|---|---|---|
| 기본 `@tool` | 없음 (타입만) | `@tool` |
| docstring 파싱 | `Args:`에 적은 설명 | `@tool(parse_docstring=True)` |
| pydantic `Field` | `Field(description=...)`에 적은 설명 | 인자 기본값에 `Field(...)` (6.3.2 `tools.py`) |

LLM은 코드를 보지 못하고 도구 설명서만 본다. 인자 설명을 붙여 두면 LLM이 인자에 무엇을 넣어야 할지 더 정확히 판단할 수 있다.